# M4 component profiling: T4 smoke first
Thin launcher for the repository CLI. No measurement formulas live here.
The first cells inspect actual GPU/driver before downloads or installation.
No canonical topology profile or gamma is produced. See `experiments/M4_COLAB.md`.


In [ ]:
from pathlib import Path
import subprocess, sys, json
EXPECTED_GPU = "T4"
RUN_ID = "CHANGE-ME-unique-t4-smoke"
REPO_URL = "https://github.com/theflyingkhui04/do-an-ptit-d22.git"
BRANCH = "feat/m4-02-02-gpu-cost-profiler"
REVISION = "SET-TO-REVIEWED-40-CHAR-COMMIT"
REPO = Path("do-an-ptit-d22")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
import torch
assert torch.cuda.is_available(), "No CUDA GPU: stop before installation/downloads"
name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print(name, capability)
expected_cap = {"T4": (7,5), "L4": (8,9)}[EXPECTED_GPU]
hardware_matches = EXPECTED_GPU in name.split() and capability == expected_cap
inspection = {"expected_gpu": EXPECTED_GPU, "observed_gpu": name,
              "gpu_compute_capability": list(capability), "canonical_hardware": hardware_matches}
Path("m4-hardware-inspection.json").write_text(json.dumps(inspection, indent=2))
assert hardware_matches, "GPU mismatch: inspection retained; stop before expensive setup"
driver = subprocess.check_output(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"], text=True).splitlines()[0]
assert int(driver.split(".")[0]) >= 580, "Pinned CUDA 13 stack requires a compatible driver; do not silently upgrade/downgrade"


In [ ]:
assert len(REVISION) == 40 and all(c in "0123456789abcdef" for c in REVISION), "Set the reviewed immutable commit"
if not REPO.exists():
    subprocess.check_call(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO)])
# Checkout refuses conflicting local changes; no reset or clean commands.
subprocess.check_call(["git", "checkout", REVISION], cwd=REPO)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True))


In [ ]:
# Isolated interpreter avoids replacing the notebook kernel's preinstalled torch.
subprocess.check_call([sys.executable, "-m", "venv", ".venv"], cwd=REPO)
PYTHON = REPO.resolve() / ".venv" / "bin" / "python"
subprocess.check_call([str(PYTHON), "-m", "pip", "install", "-r", "experiments/requirements-m4-cuda.txt"], cwd=REPO)
subprocess.check_call([str(PYTHON), "-m", "pip", "check"], cwd=REPO)


In [ ]:
assert RUN_ID != "CHANGE-ME-unique-t4-smoke", "Choose a unique run ID"
command = [str(PYTHON), "-m", "experiments.m4_cost_runner", "--run-id", RUN_ID,
           "--expected-gpu", EXPECTED_GPU, "--mode", "smoke", "--contexts", "256",
           "--batches", "1", "--k-values", "1", "--warmup", "2", "--repeats", "2"]
completed = subprocess.run(command, cwd=REPO)
print("CLI exit:", completed.returncode)


In [ ]:
ARTIFACTS = REPO / "experiments" / "runs" / "m4_cost_profiling" / RUN_ID
for filename in ("env.json", "run_manifest.json", "component_report.json"):
    path = ARTIFACTS / filename
    if path.exists():
        print(filename, path.read_text())
import shutil
archive = shutil.make_archive("m4-" + RUN_ID, "zip", root_dir=ARTIFACTS.parent, base_dir=RUN_ID)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Artifacts archive:", archive)


This smoke is not benchmark evidence. Keep every artifact, including errors and warmups.
For component-campaign commands after compatibility validation, use M4_COLAB.md.
Do not generate canonical profiles until both cost/provenance alignment and tree-runtime/model approval are resolved.
